# 协调者模式：用大模型规划，用小模型执行

## 引言

大多数智能体工作负载内部包含两类截然不同的工作：少量规划与判断，以及大量机械式的阅读与操作。网络检索是极端情形，也是本笔记本采用的示例：针对二十个事实逐一对照其权威来源进行核实，意味着要让模型读取数十万词元的网页，而按前沿模型的价格，这部分阅读费用占据主导。

协调者模式把这两类工作拆开。前沿模型负责规划检索并综合出答案，但它从不接触原始网页——廉价的工人智能体在各自并行的上下文窗口中完成全部阅读，只汇报提炼后的发现。本笔记本诚实地度量这种拆分：它在同一个问题上运行现实中的替代方案——一个拥有相同工具、被要求达到相同核实标准的前沿模型智能体——并比较真实账单与真实墙钟时间。在作者的运行中，两条路径读取的量大致相同，团队 84-98% 的输入词元按工人价格计费。

**读完这份 cookbook，你将能够：**

- 用 `multiagent` 协调者字段配置一个双模型团队：一个前沿模型协调者加若干廉价检索工人
- 通过会话事件流实时跟踪一次委派（`thread_created`、`thread_message_sent`、`thread_message_received`）
- 运行一个严谨度对齐的单前沿模型对照组，并比较真实账单
- 使用按线程给出的类型化累计 `usage` 对每个线程计量

同样的经济学适用于任何“廉价模型可以承担词元密集环节”的工作负载：文档审阅、日志分析、代码库扫描。


![Architecture of the coordinator pattern: the user's question goes to a frontier-model coordinator with no tools of its own; it sends one brief per park to parallel small-model search workers and gets distilled findings back; only the workers touch the open web, via web search and page fetch; the coordinator synthesizes the final answer.](example_data/plan_big_execute_small/architecture_diagram.png)


## 前置条件

在跟随本指南之前，请确认你已具备：

**必备知识：**
- Python 基础
- 熟悉 Managed Agents 的基本概念——智能体、环境、会话以及流式事件循环（[﷐0﷑](https://github.com/anthropics/claude-cookbooks/blob/main/managed_agents/CMA_iterate_fix_failing_tests.ipynb) 对它们都有介绍）

**必备工具：**
- Python 3.11 或更高版本
- 拥有 Managed Agents 测试版访问权限的 Anthropic API 密钥（[在此获取](https://console.anthropic.com)）

如果你还没见过 `multiagent` 字段，[﷐1﷑](https://github.com/anthropics/claude-cookbooks/blob/main/managed_agents/CMA_coordinate_specialist_team.ipynb) 用一个异构专家团队介绍了它。本笔记本使用尽可能简单的团队——只有一种工人类型——因为这里的重点是成本结构，而不是团队设计。

## 设置


In [ ]:
%%capture
%pip install -qU "anthropic>=0.121.0" python-dotenv

In [ ]:
import os
import time

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]

# The frontier model plans and synthesizes; the cheap model reads the web.
COORDINATOR_MODEL = os.environ.get("COOKBOOK_COORDINATOR_MODEL", "claude-fable-5")
WORKER_MODEL = os.environ.get("COOKBOOK_WORKER_MODEL", "claude-sonnet-5")

client = anthropic.Anthropic()

## 1. 团队：廉价的阅读者，昂贵的思考者

两份智能体定义就构成了整个团队。

**工人**——按文档的说法，是协调者可以从名册中派生的子智能体——就是一份普通的智能体定义：一个模型、一套收窄到 `web_search` + `web_fetch` 的工具集，以及一段系统提示词。每个工人实例在自己的会话线程中研究一个聚焦的子问题，因此它读取的那些巨大网页永远不会进入其他人的上下文。

**协调者**自身没有任何工具——只有一份点名工人的 `multiagent` 名册。正是这一个字段使它成为协调者：服务器自动为它提供 `create_agent`、`send_to_agent`、`wait_for_agents` 和 `list_agents`，工人同样以这种方式获得 `submit_result` 和 `send_to_parent`。你无需定义其中任何工具。

关于这种关系有两点需要知道。第一，名册在协调者被创建或更新时被快照保存——如果你修改了工人的定义，就要更新或重建协调者。第二，也是不那么显而易见的一点：**协调者看不到关于其名册智能体的任何信息**——看不到它们的提示词、名字或描述。它的 `create_agent` 工具只接受一个裸的智能体名和任务字符串。协调者对其工人的全部认知都来自它自己的系统提示词，所以请让那份描述与工人的实际提示词保持一致——服务器上没有任何机制强制这一点。（当名册只有一个智能体时，任何请求的名字都会解析到那唯一的工人；当有多种工人类型时，要在协调者的提示词中明确点名。）


In [ ]:
worker = client.beta.agents.create(
    name="search-worker",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-plan-big-execute-small"},
    model=WORKER_MODEL,
    # Everything off except the two web tools: the worker's job is
    # reading, and scoping keeps the cheap model from wandering into
    # bash or the filesystem. It's also the security boundary: workers
    # read arbitrary (untrusted) web pages, so a worker that can only
    # search, fetch, and report back is the blast radius you want for
    # that input — and the coordinator reading the reports has no
    # tools at all.
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {"enabled": False},
            "configs": [
                {"name": "web_search", "enabled": True},
                {"name": "web_fetch", "enabled": True},
            ],
        }
    ],
    system=(
        "You are a search worker researching one focused sub-question for "
        "a coordinator. Use web_search and web_fetch to find the answer. "
        "Be thorough: try multiple query phrasings, follow promising "
        "links, and cross-check facts across sources. Report back with "
        "the specific answer you found and the evidence (URLs, quotes) "
        "that supports it. If you could not find a definitive answer, say "
        "exactly what you did find and what remains uncertain. Always "
        "finish by calling submit_result."
    ),
    betas=BETAS,
)

coordinator = client.beta.agents.create(
    name="search-coordinator",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-plan-big-execute-small"},
    model=COORDINATOR_MODEL,
    multiagent={
        "type": "coordinator",
        "agents": [{"type": "agent", "id": worker.id}],
    },
    system=(
        "You are coordinating a team of search workers to answer a hard "
        "web-research question. Your workers have web_search and "
        "web_fetch; you do not. Break the question into focused "
        "sub-questions and delegate each to a worker via create_agent. "
        "Run several workers in parallel on independent sub-questions, "
        "and ALWAYS call wait_for_agents after spawning before drawing "
        "any conclusion. When a worker reports, decide whether its "
        "findings answer the sub-question or whether to send a follow-up "
        "with send_to_agent. If a worker returns an infrastructure error "
        "(rate limit, timeout) instead of findings, re-assign the same "
        "sub-question to a fresh worker. Once you have enough evidence, "
        "synthesize the workers' findings into a single final answer to "
        "the original question."
    ),
    betas=BETAS,
)

print(f"worker      {worker.id}")
print(f"coordinator {coordinator.id}")

## 2. 运行一个检索问题

为协调者创建一个环境和一个会话，把问题作为 `user.message` 发送，然后开始流式接收。会话级事件流就是协调者的主线程——整次运行的浓缩视图。工人的线程在其中以委派流量的形式出现：工人被派生时出现 `session.thread_created`，协调者把某个子问题交给它时出现 `agent.thread_message_sent`，发现汇报回来时出现 `agent.thread_message_received`。

这个问题是一项覆盖型任务——二十个事实（10 个公园 x 2 项属性），每一个都必须对照某个特定的权威来源核实。覆盖型问题正是这一模式大放异彩的地方，因为阅读是强制的：没人可以凭记忆作答，于是唯一的问题就是阅读按什么价格计费、以及它是否并行发生。（探索型问题——在巨大的搜索空间中找出一个隐藏答案，类似 [BrowseComp](https://arxiv.org/abs/2504.12516) 这类基准的风格——更依赖前沿模型的搜索直觉，因此差距会缩小。）

扇出程度取决于数据：由协调者决定派生多少工人，提示词中没有任何东西固定它的上限。下面的会话带有一个 `budget`，即对整个团队在所有线程上的总列表成本的强制上限。$10 的上限足够宽松，正常运行永远碰不到它；但如果某个糟糕的问题让协调者无休止地派生工人，会话会在上限处暂停，而不是把账单越滚越大。暂停的机制以及之后如何提高上限，见 [﷐1﷑](CMA_cap_session_spend.ipynb)。

In [ ]:
env = client.beta.environments.create(
    name="research-fanout",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
)

session = client.beta.sessions.create(
    agent=coordinator.id,
    environment_id=env.id,
    # guardrail on data-dependent fan-out: cap the whole team's list cost
    budget={"type": "limit", "max_list_cost": {"currency": "USD", "amount": "1000"}},
    betas=BETAS,
)

QUESTION = (
    "For each of the ten largest national parks in the contiguous United "
    "States by area, find: the current standard private-vehicle entrance "
    "fee, and whether the park currently requires a timed-entry or "
    "day-use reservation for peak season. Each fact must be verified "
    "against that park's official nps.gov pages (fees page and alerts/"
    "reservations page) - not from third-party summaries. Give park, fee, "
    "reservation requirement, and the nps.gov URLs you used."
)

t_start = time.monotonic()

client.beta.sessions.events.send(
    session.id,
    betas=BETAS,
    events=[{"type": "user.message", "content": [{"type": "text", "text": QUESTION}]}],
)


def text_of(content):
    return "".join(b.text for b in content or [] if b.type == "text")


def clip(s, n=160):
    return s[:n] + ("..." if len(s) > n else "")


final_answer = ""
with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        match ev.type:
            case "agent.message":
                if text := text_of(ev.content).strip():
                    final_answer = text
                    print(f"[coordinator] {clip(text, 200)}")
            case "session.thread_created":
                print(f"[spawn] {ev.agent_name} ({ev.session_thread_id})")
            case "agent.thread_message_sent":
                print(f"[delegate -> {ev.to_agent_name}] {clip(text_of(ev.content))}")
            case "agent.thread_message_received":
                print(f"[report <- {ev.from_agent_name}] {clip(text_of(ev.content))}")
            case "session.status_idle":
                break

print(f"\n[team finished in {time.monotonic() - t_start:.0f}s]")
print("=" * 70)
print(final_answer)

值得注意的形态是：每一条 `[delegate ->]` 行都是一条小消息，每一条 `[report <-]` 行都是一份提炼后的摘要。产生这些汇报的那几兆字节搜索结果与抓取页面，从未越过协调者的上下文。这一隔离就是整个成本故事。为了公平定价，下一节运行现实中的替代方案，然后我们对两者计量。


## 3. 运行对照组：一个前沿模型智能体，同样的核实标准

如果没有这个模式，这会花多少钱？现实中的替代方案是一个拥有同样两个网络工具的单前沿模型智能体。有一个细节决定了这个比较是有价值还是一文不值：**单智能体必须被要求达到同样的核实标准。** 若任其自行判断，前沿模型是很节省的——每个事实只读一个来源，费用很低，但那是严谨度更低的产物，而不是同一工作在另一个价格上的版本。所以下面的单智能体提示词要求团队本来就在做的事：每个事实由两次独立抓取核实，冲突要复核并标注。

同样的问题，安静的流式输出。


In [ ]:
solo = client.beta.agents.create(
    name="solo-researcher",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-plan-big-execute-small"},
    model=COORDINATOR_MODEL,
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {"enabled": False},
            "configs": [
                {"name": "web_search", "enabled": True},
                {"name": "web_fetch", "enabled": True},
            ],
        }
    ],
    system=(
        "You research hard web questions with audit-grade rigor. Use "
        "web_search and web_fetch. For EVERY fact you report, verify it "
        "from at least two independent fetches (the authoritative page "
        "plus one corroborating source), and re-fetch when two sources "
        "disagree. Never carry a fact forward on one source or from "
        "memory. In your answer, give each fact with both source URLs, "
        "and explicitly flag any fact where sources conflicted. Before "
        "finishing, audit your own answer: list each claim and check it "
        "has two cited sources."
    ),
    betas=BETAS,
)

t_solo = time.monotonic()
solo_session = client.beta.sessions.create(agent=solo.id, environment_id=env.id, betas=BETAS)
client.beta.sessions.events.send(
    solo_session.id,
    betas=BETAS,
    events=[{"type": "user.message", "content": [{"type": "text", "text": QUESTION}]}],
)

solo_answer = ""
with client.beta.sessions.events.stream(solo_session.id, betas=BETAS) as stream:
    for ev in stream:
        match ev.type:
            case "agent.message":
                if text := text_of(ev.content).strip():
                    solo_answer = text
            case "session.status_idle":
                break

print(f"[solo finished in {time.monotonic() - t_solo:.0f}s]")
print(clip(solo_answer, 300))

## 4. 对两次运行计量并定价

成本归因是 API 内置的能力。每个会话线程都带有累计 `usage`，会话和每个线程都报告 `usage.list_cost`：按公开列表价格计算的词元成本，由服务器端计算。列出线程，把主线程（`parent_thread_id is None`）作为协调者，子线程就是工人——单智能体会话根本没有子线程。会话自身的 `list_cost` 就是总额，因此核心数字无需维护任何价格表。

（如果你需要逐请求的细节，它在每个线程自己的事件流里——会话级事件流只携带主线程的 `span.model_request_end` 事件。）

有一个数字 API 无法直接给你，那就是反事实：这支完全相同的团队工作负载若全部按前沿模型价格计费会是多少。这需要把一张价格表套用到词元数量上，因此代码为这一个“假设”场景保留了来自[价格页面](https://platform.claude.com/docs/en/about-claude/pricing)的输入与输出价格——5 分钟缓存写入按输入价格的 1.25 倍计费，1 小时写入按 2 倍，缓存读取按 0.1 倍——而所有真实数字都使用 `list_cost`。

In [ ]:
# $ / MTok input and output from the pricing page, used only for the
# all-frontier counterfactual below; the real bills come from the API's
# server-side list_cost. Sonnet 5 shows its introductory rate ($2/$10
# through Aug 31, 2026; standard $3/$15 after — update these then).
PRICES = {
    "claude-fable-5": {"input": 10.0, "output": 50.0},
    "claude-sonnet-5": {"input": 2.0, "output": 10.0},
}


def total_input(u):
    cache = u.cache_creation  # None on threads with no cache activity
    return (
        u.input_tokens
        + u.cache_read_input_tokens
        + (cache.ephemeral_5m_input_tokens if cache else 0)
        + (cache.ephemeral_1h_input_tokens if cache else 0)
    )


def counterfactual_cost(u, model):
    """Re-price a thread's tokens at another model's rates (the what-if)."""
    p = PRICES[model]
    cache = u.cache_creation
    return (
        u.input_tokens * p["input"]
        + (cache.ephemeral_5m_input_tokens if cache else 0) * p["input"] * 1.25
        + (cache.ephemeral_1h_input_tokens if cache else 0) * p["input"] * 2.0
        + u.cache_read_input_tokens * p["input"] * 0.1
        + u.output_tokens * p["output"]
    ) / 1e6


def dollars(list_cost):
    return float(list_cost.amount) / 100  # amount is an integer string in cents


def report(session_id):
    session_usage = client.beta.sessions.retrieve(session_id, betas=BETAS).usage
    threads = list(client.beta.sessions.threads.list(session_id, betas=BETAS))
    primary = next(t for t in threads if t.parent_thread_id is None)
    workers = [t for t in threads if t.parent_thread_id is not None]
    workers_in = sum(total_input(t.usage) for t in workers)
    print(
        f"  primary thread ({primary.agent.model.id}): "
        f"{total_input(primary.usage):>9,} in / {primary.usage.output_tokens:>6,} out"
        f"  -> ${dollars(primary.usage.list_cost):.2f}"
    )
    if workers:
        print(
            f"  {len(workers)} worker(s): {workers_in:>9,} in / "
            f"{sum(t.usage.output_tokens for t in workers):>6,} out"
            f"  -> ${sum(dollars(t.usage.list_cost) for t in workers):.2f}"
        )
        print(
            f"  workers' share of input: {workers_in / (workers_in + total_input(primary.usage)):.0%}"
        )
    total = dollars(session_usage.list_cost)
    print(f"  total cost (session usage.list_cost): ${total:.2f}")
    return total, threads


print("split team (fable coordinator + sonnet workers):")
split_cost, split_threads = report(session.id)

print("\nsolo frontier agent:")
solo_cost, _ = report(solo_session.id)

# The counterfactual that isolates the rate split: this run's team
# workload with every token billed at the frontier rate.
frontier_team_cost = sum(counterfactual_cost(t.usage, COORDINATOR_MODEL) for t in split_threads)

print(f"\nsolo / split cost ratio on this pair of runs: {solo_cost / split_cost:.1f}x")
print(f"the split team's workload at all-frontier rates: ${frontier_team_cost:.2f}")

两次运行的阅读量几乎相同——这正是对齐核实标准的意义所在。不同的是阅读计费的价格和工作形态：团队的二十次查询以廉价价格的并行工人线程运行，而单智能体在一个按前沿模型价格计费的上下文里逐一串行完成。在作者的运行中，团队 84-98% 的输入词元按工人价格计费。词元数量每次运行都不同，所以任何一次打印出的比例都只能当作一个样本；结构才是稳定的部分。

四个诚实的注意事项，都是在编写本笔记本过程中观察到的：

* **比较必须建立在严谨度对齐之上。** 任其自行判断的单前沿模型智能体阅读量小得多（每个事实一个来源），费用也比团队更低——但那是另一个严谨度更低的产品。只有在核实标准固定的前提下才比较两条路径。
* **委派存在成本下限。** 每个工人线程都要支付固定的启动开销。把同样的工作拆成更多、更窄的任务简报，反而让账单升高而不是降低——简报粒度存在最优值。
* **核实标准只覆盖你写进去的内容。** 已提交运行中的两条路径都对照 nps.gov 核实了全部二十个事实——但两者都凭模型记忆构建公园清单，于是把 Kings Canyon 放进了本应属于 Great Smoky Mountains 的第 #10 位（Kings Canyon 按面积是 #12）。事实被审计了，问题的拆解却没有。如果前提重要，就多花一次委派，让一个工人去核实它。
* **协调者只知道你告诉它的东西。** 服务器上没有任何东西能让它看到工人的提示词（见第 1 节），因此这套经济学也取决于你在协调者的提示词中准确描述工人的行为。

什么情况下这种拆分不划算？在狭窄的问题上，阅读量太小，没有可套利的空间。如果协调者凭自身知识作答（没有委派），你就白白支付了一次前沿模型的往返——要留意那些没有 `[spawn]` 行的运行。而如果任务需要对原始材料本身作出前沿模型的判断（是细腻的分析文档，而非查找事实），廉价阅读者可能恰恰把最重要的内容摘要掉了。

## 回顾

在本指南中，你构建了多智能体团队最有用的最简形态，并度量了它带来的收益：

* **配置了一个双模型团队**——一个唯一能力就是 `multiagent` 名册的前沿模型协调者，以及一个工具范围限定为网络工具的廉价工人
* **通过会话流上的 `thread_created` / `thread_message_sent` / `thread_message_received` 实时跟踪了委派**
* **运行了严谨度对齐的单前沿模型对照组**，并比较真实账单与真实墙钟时间，而不是估算
* **对每个线程计量**，使用每个会话线程都携带的类型化累计 `usage`（`session.usage` 是整个团队的总额）

要进一步深入：

1. 增加带有受限工具集的专家型工人类型——[﷐0﷑](https://github.com/anthropics/claude-cookbooks/blob/main/managed_agents/CMA_coordinate_specialist_team.ipynb) 展示了一个三角色团队，以及按角色限定工具范围为何重要
2. 把按线程计量纳入你的生产遥测：线程级 usage 是你按每次委派（而不只是按会话）归因花费的方式
3. 在你自己词元密集的工作负载上尝试同样的拆分——文档审阅和日志分诊与网络检索具有相同的阅读密集、覆盖型特征

参考：[多智能体会话文档](https://platform.claude.com/docs/en/managed-agents/multi-agent)。
